In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10

# Datasets and DataLoader 

In [14]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.5,0.5,0.5],
        std = [0.5,0.5,0.5]
    )
])
trainset = CIFAR10(root="./data",train=True,download=True,transform=transform)
testset = CIFAR10(root="./data",train=False,download=True,transform=transform)

100%|██████████████████████████████████████| 170M/170M [1:16:42<00:00, 37.0kB/s]


# Build the CNN

In [17]:
trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

In [20]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1)
        x = self.fc_layers(x)
        return x
    

In [21]:
model = CNN()

In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training The CNN

In [23]:
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0.0
    for images,labels in trainloader:
        optimizer.zero_grad()
        output = model.forward(images) #Forward propagation
        loss = criterion(output,labels) #Loss
        loss.backward() #Backward propagation
        optimizer.step() #Update params
        epoch_training_loss+=loss.item()
    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")
        
        
        

epoch=1/10 & loss=1.3924130006214541
epoch=2/10 & loss=0.9678865554540054
epoch=3/10 & loss=0.7666289977862707
epoch=4/10 & loss=0.6446496225760111
epoch=5/10 & loss=0.5416682886002618
epoch=6/10 & loss=0.45412028733345555
epoch=7/10 & loss=0.37240429207339615
epoch=8/10 & loss=0.29746776218514154
epoch=9/10 & loss=0.23618367139983665
epoch=10/10 & loss=0.1851048286280135


# Evaluate The CNN

In [24]:
total_labels=0
correct_labels=0
for images,labels in testloader:
    output = model.forward(images)
    _,predicted = torch.max(output,1)
    correct_labels += (predicted==labels).sum().item()
    total_labels += labels.size(0)
print(f"Accurcay = {(correct_labels/total_labels)*100}")

Accurcay = 75.55
